## Biweekly 20.01

In [12]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.io as pio
from scipy import stats
from scipy.stats import chi2_contingency
from typing import Callable

def _resolve_project_root() -> Path:
    """locate project root containing config.py."""
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'config.py').exists():
            print(candidate)
            return candidate
    raise FileNotFoundError('config.py not found in cwd or parents')


PROJECT_ROOT = _resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from config import GENE_PATHS, SOURCE_PALETTE, VARIANT_PATHS

/Users/markus/in-silico-vg-analysis


In [13]:
CLINGEN_VAR = VARIANT_PATHS['clingen']
BG_VAR = VARIANT_PATHS['background']
BG_NULL_VAR = VARIANT_PATHS['background_null']
CLINGEN_NULL_VAR = VARIANT_PATHS['clingen_null']

CLINGEN_GENE = GENE_PATHS['clingen']
BG_GENE = GENE_PATHS['background']
BG_NULL_GENE = GENE_PATHS['background_null']
CLINGEN_NULL_GENE = GENE_PATHS['clingen_null']

PATHS = {
    'background': BG_GENE,
    'background_null': BG_NULL_GENE,
    'clingen': CLINGEN_GENE,
    'clingen_null': CLINGEN_NULL_GENE,
}

df = pl.read_parquet(CLINGEN_NULL_GENE)
print(df.columns)
print(df.head())

# TODO: add dedublication logic

def _dedup_scores_by_variant(
    path: str | Path, label: str, columns: list[str] | None = None
) -> pl.DataFrame:
    """dedup track-level scores to one row per variant (max abs score).

    args:
        path (str | Path): variant file path.
        label (str): source label to attach.
        columns (list[str] | None): extra columns to retain after dedup.

    returns:
        pl.DataFrame: deduped variants with selected columns.

    raises:
        FileNotFoundError: when the file is missing.
        ValueError: when required columns are missing.
    """
    columns = [] if columns is None else columns

    file_path = Path(path)
    if not file_path.exists():
        raise FileNotFoundError(f'missing variant file: {file_path}')

    schema_cols = pl.read_parquet(file_path, n_rows=0).columns
    if 'raw_score' in schema_cols:
        score_col = 'raw_score'
    elif 'score' in schema_cols:
        score_col = 'score'
    else:
        raise ValueError(f'missing raw_score/score column in {file_path}')

    group_cols = [
        c for c in ['variant_id', 'CHROM', 'POS', 'REF', 'ALT', 'gene_id'] if c in schema_cols
    ]
    if not group_cols:
        raise ValueError('no variant identifier columns found')

    missing = [c for c in columns if c not in schema_cols]
    if missing:
        raise ValueError(f'missing columns in {file_path}: {missing}')

    select_cols = list(dict.fromkeys([score_col, *group_cols, *columns]))

    deduped = (
        pl.scan_parquet(file_path)
        .select(select_cols)
        .filter(pl.col(score_col).is_not_null())
        .with_columns(pl.col(score_col).abs().alias('_abs_score'))
        .sort(group_cols + ['_abs_score'], descending=[False] * len(group_cols) + [True])
        .unique(subset=group_cols, keep='first')
        .with_columns(
            pl.col(score_col).alias('raw_score'),
            pl.col(score_col).abs().alias('abs_score'),
            pl.lit(label).alias('source'),
        )
        .drop('_abs_score')
        .select(
            list(dict.fromkeys([*group_cols, *columns, 'raw_score', 'abs_score', 'source']))
        )
        .collect()
    )

    return deduped



def _load_variant_table(path: str | Path, label: str, columns: list[str] | None) -> pl.DataFrame:
    """load selected columns from a variant file and tag source after dedup.

    args:
        path (str | Path): variant file path.
        label (str): source label to attach.
        columns (list[str] | None): columns to load from the file.

    returns:
        pl.DataFrame: deduped subset with source column attached.

    raises:
        FileNotFoundError: when the file is missing.
        ValueError: when the file is empty or missing columns.
    """
    columns = [] if columns is None else columns

    table = _dedup_scores_by_variant(path=path, label=label, columns=columns)
    if table.is_empty():
        raise ValueError(f'no rows in variant file: {Path(path)}')

    return table


def _load_scores(path: str | Path, label: str) -> pl.DataFrame:
    """load deduped scores with source tag for downstream analysis."""
    return _load_variant_table(path=path, label=label, columns=None)

['gene_id', 'n_variants', 'vg_predicted', 'vg_predicted_perm', 'n_variants_perm', 'sum_sq_raw_score', 'mean_raw_score', 'mean_abs_effect', 'median_abs_effect', 'std_abs_effect', 'min_abs_effect', 'max_abs_effect', 'skewness_effect', 'q90_abs_effect', 'min_variant_id', 'min_variant_score', 'max_variant_id', 'max_variant_score', 'mean_dist_to_tss', 'median_dist_to_tss', 'min_dist_to_tss', 'max_dist_to_tss', 'n_high_impact_gt05', 'n_high_impact_gt1', 'mean_abs_promoter', 'n_variants_promoter', 'mean_abs_up2kb', 'n_variants_up2kb', 'mean_abs_up10kb', 'n_variants_up10kb', 'mean_abs_up100kb', 'n_variants_up100kb', 'mean_abs_down2kb', 'n_variants_down2kb', 'mean_abs_gene_body', 'n_variants_gene_body', 'gene_symbol', 'gene_type', 'chrom', 'genomic_length', 'exonic_length', 'coding_length', 'intronic_length', 'utr_length', 'utr5_length', 'mane_transcript_id', 'is_mane', 'tpm_muscle', 'ncRVIS', 'loeuf_score', 'ncGERP', 'RVIS_score', 'ncCADD', 'pHaplo', 'pTriplo', 'Episcore', 'pLI', 'median_tpm',

In [14]:
CLINGEN_TSV = Path(
    '/Users/markus/university/ML-in-biotech-CB206V-ws25/data/intermediate/'
    'dataset3_ClinGen/ClinGen_variants.tsv'
)
BG_TSV = Path(
    '/Users/markus/university/ML-in-biotech-CB206V-ws25/data/intermediate/'
    'dataset4_background/background_variants.tsv'
)


def _load_ac_table(path: str | Path, label: str) -> pl.DataFrame:
    """load variant tsv with ac and gene_id for matching.

    args:
        path (str | Path): tsv file path.
        label (str): source label for error messages.

    returns:
        pl.DataFrame: columns variant_id, AC, gene_id.

    raises:
        FileNotFoundError: when the file is missing.
        ValueError: when required columns are missing or the file is empty.
    """
    file_path = Path(path)
    if not file_path.exists():
        raise FileNotFoundError(f'missing tsv file for {label}: {file_path}')

    table = pl.read_csv(file_path, separator='\t')
    required = ['variant_id', 'AC', 'gene_tag']
    missing = [col for col in required if col not in table.columns]
    if missing:
        raise ValueError(f'missing columns in {file_path}: {missing}')

    table = (
        table
        .with_columns(
            pl.col('gene_tag').str.split('|').list.get(0).alias('gene_id')
        )
        .select(['variant_id', 'AC', 'gene_id'])
    )
    if table.is_empty():
        raise ValueError(f'no rows in tsv file: {file_path}')

    return table


gene_tables = {
    'clingen': pl.read_parquet(CLINGEN_GENE),
    'background': pl.read_parquet(BG_GENE),
    'background_null': pl.read_parquet(BG_NULL_GENE),
    'clingen_null': pl.read_parquet(CLINGEN_NULL_GENE),
}

for name, table in gene_tables.items():
    print(f'{name} gene shape: {table.shape}')

variant_tables = {
    'clingen': _load_scores(path=CLINGEN_VAR, label='clingen'),
    'background': _load_scores(path=BG_VAR, label='background'),
    'background_null': _load_scores(path=BG_NULL_VAR, label='background_null'),
    'clingen_null': _load_scores(path=CLINGEN_NULL_VAR, label='clingen_null'),
}

for name, table in variant_tables.items():
    print(f'{name} variant shape: {table.shape}')

ac_tables = {
    'clingen': _load_ac_table(path=CLINGEN_TSV, label='clingen'),
    'background': _load_ac_table(path=BG_TSV, label='background'),
}

matched_variants = {}
for name in ['clingen', 'background']:
    variants = variant_tables[name]
    if 'gene_id' not in variants.columns:
        raise ValueError(f'missing gene_id in variant table: {name}')

    ac_table = ac_tables[name].rename({'gene_id': 'gene_id_tsv', 'AC': 'ac'})
    joined = variants.join(ac_table, on='variant_id', how='left')

    missing_ac = joined.filter(pl.col('ac').is_null()).height
    coverage = 1 - (missing_ac / joined.height)
    print(f'{name} ac coverage: {coverage:.2%}')

    matched_variants[name] = joined.drop('gene_id_tsv')


clingen gene shape: (316, 66)
background gene shape: (349, 66)
background_null gene shape: (349, 68)
clingen_null gene shape: (316, 68)
clingen variant shape: (1743183, 9)
background variant shape: (1999142, 9)
background_null variant shape: (2460730, 9)
clingen_null variant shape: (2165642, 9)
clingen ac coverage: 100.00%
background ac coverage: 100.00%


In [ ]:
"""
TODO: 

From variant level files, calculate 

	1.	top1_frac — fraction of V_g explained by the single top variant (largest v_i).
	2.	top3_frac — fraction of V_g explained by the top 3 variants (or fewer if gene has <3 variants).
	3.	top10_frac — fraction of V_g explained by the top 10 variants (or fewer).
	4.	frac_variants_for_90 — minimal fraction of variants (k / n_variants) needed to reach ≥ 90% of V_g.
	5.	frac_variants_for_99 — minimal fraction of variants needed to reach ≥ 99% of V_g.
	6.	count_variants_for_90, count_variants_for_99 — the raw counts k (optional, but often useful).

You can find for Clingen and Background vg_predicted column in variant level files and for Synthetic dataset you can find vg_predicted_perm column.
plot them side by side for each dataset to see how they compare. add test statistics to see if they are different.
"""

# What's new this week

1. We will change the naming for NULL sets to synthetic (SYNTH) and specify that our ClinGen dataset is Haploinsufficent. Therefore the new naming from here on will be. The reason to do this is because NULL is too vauge description for our usecase. 

- ClinGen_HI_Gnomad 
- ClinGen_HI_Synth
- Background_Gnomad
- Background_Sytnh

2. We will introduce VG scores to our Synthetic datasets. We do this by  comparing observed per-gene genetic variance to a permutation-based null distribution generated by randomly reassigning allele frequencies (AFs) from a global pool of observed variants keeping the distingtion between ClinGen_HI and Background datasets. Our new variants file have now new column vg_predicted_perm

3. For our gene level analysis we divide genes by different categories based on if the gene is singleton or doubleton or tripleton. We make extra annotations to our vairant level files on from AC values from intermediate variant files and calculate with aggregator module fractions of how many variants are affecting the gene level absolute score.

4. We calculate the gene-metrics correlations between whole groups, not just feature based

In [ ]:
# TODO: add Violin plots for:

# 